# RescueVision v2 - xBD native preprocessing -> single tar

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from rescuevision.data.xbd_parser import parse_xbd_json
from rescuevision.data.xbd_preprocess import generate_masks_for_tile, generate_crops_for_tile
from rescuevision.data.split_utils import generate_event_aware_splits
from rescuevision.utils.io import find_image_pairs
from rescuevision.utils.image import load_image

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False
from scipy.ndimage import binary_dilation
print("cv2 available:", HAS_CV2)

print("kaggle inputs:", os.listdir("/kaggle/input"))
RAW = None
for cand in Path("/kaggle/input").rglob("labels"):
    if cand.is_dir() and (cand.parent / "images").is_dir():
        RAW = cand.parent
        break
assert RAW is not None, "could not find a dir with images/ + labels/ under /kaggle/input"
print("raw xBD root:", RAW)
mark("preproc:raw-found", extra=str(RAW))

OUT = Path("/tmp/processed_xbd")     # native resolution this time; only the tar lands in working
CROP_MARGIN = 16

for sub in ["images/pre", "images/post", "masks/building_binary", "masks/damage_multiclass",
            "masks/damage_multiclass_dil", "masks/change_binary",
            "crops/building_pre", "crops/building_post", "metadata"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

pairs = []
for _r in ROOTS:
    pairs.extend(find_image_pairs(_r / "images"))
_seen = set(); _uniq = []
for _pre, _post in pairs:
    _tid = _pre.stem.replace("_pre_disaster", "")
    if _tid not in _seen:
        _seen.add(_tid); _uniq.append((_pre, _post))
pairs = _uniq
print(f"found {len(pairs)} unique pre/post image pairs")
mark("preproc:pairs", extra=str(len(pairs)))

In [ ]:
tile_rows, building_rows = [], []
crop_counter = 0
processed = 0
written_bytes = 0
n_shifted = 0

def _png(path, arr):
    global written_bytes
    Image.fromarray(arr).save(path)
    written_bytes += path.stat().st_size

def _dilate_minor(dmg, iters=1):
    """xView2-winner trick: annotators under-mark damage edges -> grow minor/major
    into background only (never overwrite another class)."""
    out = np.asarray(dmg).copy()
    grow = binary_dilation((out == 2) | (out == 3), iterations=iters) & (out == 0)
    out[grow] = 2
    return out.astype(np.uint8)

def _phase_shift(pre, post):
    """(dy,dx) so np.roll(pre,(dy,dx)) aligns with post. Self-verifying: if the roll
    does not shrink the residual shift, flip sign; if still bad -> (0,0)."""
    if not HAS_CV2:
        return 0, 0
    try:
        g1 = np.float32(np.array(Image.fromarray(pre).convert("L")))
        g2 = np.float32(np.array(Image.fromarray(post).convert("L")))
        h, w = g1.shape
        win1 = cv2.createHanningWindow((w, h), cv2.CV_32F)
        dx, dy = cv2.phaseCorrelate(g1, g2, win1)
        dy, dx = int(round(dy)), int(round(dx))
        if abs(dy) > 8 or abs(dx) > 8:
            return 0, 0
        def _resid(y, x):
            r = np.roll(g1, (y, x), axis=(0, 1))
            rx, ry = cv2.phaseCorrelate(r, g2, win1)
            return abs(ry) + abs(rx)
        r0 = _resid(dy, dx)
        if r0 >= abs(dy) + abs(dx) and (dy or dx):
            r1 = _resid(-dy, -dx)
            if r1 < r0:
                dy, dx = -dy, -dx
            else:
                return 0, 0
        return dy, dx
    except Exception:
        return 0, 0

for pre_path, post_path in tqdm(pairs, desc="processing tiles", miniters=25):
    tile_id = pre_path.stem.replace("_pre_disaster", "")
    try:
        pre_tile = parse_xbd_json(RAW / "labels" / f"{tile_id}_pre_disaster.json", pre_path)
        post_tile = parse_xbd_json(RAW / "labels" / f"{tile_id}_post_disaster.json", post_path)
        pre_image = load_image(pre_path)
        post_image = load_image(post_path)
    except Exception as e:
        print(f"skip {tile_id}: {e}")
        continue

    h, w = pre_image.shape[:2]
    masks = generate_masks_for_tile(pre_tile, post_tile, h, w)

    _png(OUT / "images/pre" / f"{tile_id}.png", pre_image)
    _png(OUT / "images/post" / f"{tile_id}.png", post_image)
    dmg = np.asarray(masks["damage_multiclass"]).astype(np.uint8)
    _png(OUT / "masks/building_binary" / f"{tile_id}.png", masks["building_binary"])
    _png(OUT / "masks/damage_multiclass" / f"{tile_id}.png", dmg)
    _png(OUT / "masks/damage_multiclass_dil" / f"{tile_id}.png", _dilate_minor(dmg))
    _png(OUT / "masks/change_binary" / f"{tile_id}.png", masks["change_binary"])

    sdy, sdx = _phase_shift(pre_image, post_image)
    if sdy or sdx:
        n_shifted += 1

    disaster_name = tile_id.rsplit("_", 1)[0] if "_" in tile_id else tile_id
    try:
        import json as _json
        _meta = _json.loads((LAB / f"{tile_id}_post_disaster.json").read_text()).get("metadata", {})
        disaster_type = _meta.get("disaster_type", "") or ""
    except Exception:
        disaster_type = ""

    tile_rows.append({
        "tile_id": tile_id,
        "disaster_name": disaster_name,
        "disaster_type": disaster_type,
        "num_buildings": pre_tile.num_buildings,
        "post_num_buildings": post_tile.num_buildings if post_tile else 0,
        "shift_dy": sdy,
        "shift_dx": sdx,
    })

    if post_tile is not None:
        for crop in generate_crops_for_tile(pre_tile, post_tile, pre_image, post_image,
                                            margin=CROP_MARGIN):
            if crop["damage_id"] is None or crop["damage_id"] == 255:
                continue
            crop_id = f"crop_{crop_counter:07d}"
            crop_counter += 1
            _png(OUT / "crops/building_pre" / f"{crop_id}.png", crop["pre_crop"])
            _png(OUT / "crops/building_post" / f"{crop_id}.png", crop["post_crop"])
            building_rows.append({
                "crop_id": crop_id,
                "tile_id": tile_id,
                "label": crop["damage_label"],
                "label_id": crop["damage_id"],
                "area": crop["area"],
                "disaster_name": crop["disaster_name"],
                "disaster_type": crop["disaster_type"],
                "pre_crop_path": f"crops/building_pre/{crop_id}.png",
                "post_crop_path": f"crops/building_post/{crop_id}.png",
            })

    processed += 1
    if processed % 200 == 0:
        mark("preproc:progress", extra=f"{processed}/{len(pairs)} tiles, {written_bytes/1e9:.2f}GB, {n_shifted} shifted")

print(f"processed {processed}/{len(pairs)} | {len(building_rows)} crops | {written_bytes/1e9:.2f} GB | shifted tiles: {n_shifted}")
mark("preproc:tiles-done", extra=f"{processed} tiles, {n_shifted} shifted, {written_bytes/1e9:.2f}GB")

In [ ]:
tiles_df = pd.DataFrame(tile_rows)
tiles_df.to_csv(OUT / "metadata/tiles.csv", index=False)

buildings_df = pd.DataFrame(building_rows)
buildings_df.to_csv(OUT / "metadata/buildings.csv", index=False)

splits = generate_event_aware_splits(
    OUT / "metadata/tiles.csv", OUT / "metadata/splits_event_aware.csv")

print(splits["split"].value_counts())
print("buildings per damage class:")
print(buildings_df["label"].value_counts())
mark("preproc:splits", extra=str(splits["split"].value_counts().to_dict()))

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

tid = tiles_df.iloc[0]["tile_id"]
img = np.array(Image.open(OUT / "images/post" / f"{tid}.png"))
dmg = np.array(Image.open(OUT / "masks/damage_multiclass" / f"{tid}.png"))
dmgd = np.array(Image.open(OUT / "masks/damage_multiclass_dil" / f"{tid}.png"))
bld = np.array(Image.open(OUT / "masks/building_binary" / f"{tid}.png"))
print("tile0 shape:", img.shape)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img); axes[0].set_title(f"post {tid} (native)")
axes[1].imshow(bld, cmap="gray"); axes[1].set_title("building")
axes[2].imshow(dmg, vmin=0, vmax=4, cmap="turbo"); axes[2].set_title("damage")
axes[3].imshow(dmgd, vmin=0, vmax=4, cmap="turbo"); axes[3].set_title("damage (dilated)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / "sanity_tile.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
import subprocess, json as _json
TAR_PATH = WORK / "processed_xbd_native.tar"
subprocess.run(["tar", "-cf", str(TAR_PATH), "-C", "/tmp", "processed_xbd"], check=True)
n_files = sum(1 for _ in OUT.rglob("*") if _.is_file())
summary = {
    "tiles": int(len(tiles_df)),
    "building_crops": int(len(buildings_df)),
    "shifted_tiles": int(n_shifted),
    "tree_files": int(n_files),
    "tree_gb": round(written_bytes / 1e9, 2),
    "tar_gb": round(TAR_PATH.stat().st_size / 1e9, 2),
    "splits": splits["split"].value_counts().to_dict(),
    "damage_crop_counts": buildings_df["label"].value_counts().to_dict(),
}
(WORK / "preprocess_summary.json").write_text(_json.dumps(summary, indent=2))
print(_json.dumps(summary, indent=2))
print("working dir:", sorted(p.name for p in WORK.iterdir()))
mark("preproc:tar-done", extra=f"{summary['tar_gb']}GB")